# 📊 Exploratory Data Analysis: Telecom Customer Churn

This notebook demonstrates the exploratory research phase that helped uncover patterns in the dataset. The insights found here led directly to the decisions made in the `DataPreprocessor` class and modeling ensemble of our MLOps pipeline.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')

# Configure aesthetics
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (10, 5)


## 1. Data Loading & Initial Inspection


In [ ]:
df = pd.read_csv('data/WA_Fn-UseC_-Telco-Customer-Churn.csv')
display(df.head())

display(df.info())


## 2. Uncovering the `TotalCharges` Bug
During initial inspection, `TotalCharges` shows up as an object (string) instead of a float. Let's find out why.


In [ ]:
# Convert to numeric, forcing errors to NaN. Then isolate the NaNs.
df['TotalCharges_numeric'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
missing_charges = df[df['TotalCharges_numeric'].isnull()]

display(missing_charges[['tenure', 'MonthlyCharges', 'TotalCharges']])
print(f"\nCustomers with blank TotalCharges have tenure: {missing_charges['tenure'].unique()}")


> **🔍 Discovery**: Customers with `tenure = 0` have blank spaces `' '` for `TotalCharges` because they are brand new and haven't been billed for a full total yet.
>
> **🚀 Pipeline Action**: In `data_preprocessing.py`, we explicitly coerce this column using `pd.to_numeric(errrors='coerce')` and impute the median, which handles this silently in production.


## 3. Class Imbalance Analysis
Let's check the distribution of the target variable to decide on our metric strategy.


In [ ]:
plt.figure(figsize=(6,6))
ax = sns.countplot(data=df, x='Churn')
plt.title('Target Variable Distribution (Class Imbalance)')
plt.show()

print("Class Proportions:")
print(df['Churn'].value_counts(normalize=True) * 100)


> **🔍 Discovery**: The target classes are heavily imbalanced (73% No-Churn, 27% Churn).
>
> **🚀 Pipeline Action**: Evaluating by accuracy will be very misleading here. We configured the pipeline and MLflow tracking to prioritize **ROC AUC** and **F1 Score**. During model tuning, we also inject `class_weight='balanced'` for Logistic Regression and Random Forest, and `scale_pos_weight` for XGBoost to heavily penalize missing the minority class.


## 4. Analyzing Tenure Patterns (Non-Linearity)


In [ ]:
plt.figure(figsize=(12, 6))
sns.histplot(data=df, x='tenure', hue='Churn', multiple='stack', bins=72)
plt.title('Churn Count by Tenure Length')
plt.show()


> **🔍 Discovery**: The churn rate is massively concentrated in the first 1-12 months. After 24-48 months, customers become highly loyal.
>
> **🚀 Pipeline Action**: We explicitly engineered the `tenure_group` bucketed categorical feature. This feeds the non-linear, temporal step-function logic cleanly into our tree models.


## 5. Contract Types: The Strongest Predictor


In [ ]:
plt.figure(figsize=(8, 5))
sns.countplot(data=df, x='Contract', hue='Churn')
plt.title('Absolute Churn by Contract Type')
plt.show()


> **🔍 Discovery**: Month-to-month contracts account for almost all churn. Customers on 1 and 2-year contracts are extremely safe and exhibit completely different behaviors.
>
> **🚀 Pipeline Action**: We created the engineered `is_month_to_month` binary feature to make this dominant signal immediately accessible to Logistic Regression.


## 6. Financial Intensity: Monthly Charges vs Tenure


In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(data=df, x='tenure', y='MonthlyCharges', hue='Churn', alpha=0.4)
plt.title('Tenure vs Monthly Charges (colored by Churn)')
plt.show()


> **🔍 Discovery**: High monthly charges combined with low tenure equates to maximum churn risk. The data clearly isolates a grouping of 'expensive, new' customers who drop off quickly.
>
> **🚀 Pipeline Action**: We introduced the proxy feature `charges_per_month = TotalCharges / (tenure + 1)` and the `is_high_value` flags. These computed domains add interaction variables implicitly, increasing our F1-score out of the box.
